# GuacaMol Offline Design Experiment Tutorial

This tutorial demonstrates how to run an offline active learning design experiment using the **GuacaMol** molecular dataset. We target the **QED (Quantitative Estimate of Drug-likeness)** property and use the `split_mode="low_vs_high"` split to create a realistic cold-start scenario.

### What is GuacaMol?

[GuacaMol](https://github.com/BenevolentAI/guacamol) is a benchmark suite for goal-directed molecular generation. It contains a corpus of ~1.6 million drug-like SMILES strings drawn from ChEMBL. Each molecule can be characterised by physicochemical properties (such as QED, molecular weight, logP) computed via RDKit.

In this tutorial we work with a small subset (`max_molecules=1000`) so the notebook runs quickly.

### What does `split_mode="low_vs_high"` do?

The `split_mode="low_vs_high"` option seeds the **training set with only low-scoring molecules** (the bottom half of the QED distribution), while placing the high-scoring molecules in the candidate pool. This creates a **cold-start active learning scenario**: the surrogate model must learn to identify high-QED molecules despite having been trained exclusively on low-QED examples. It is a deliberately hard and realistic setting — analogous to a drug-discovery project where the initial screening library is biased toward less drug-like compounds.

### How does the MLP surrogate use precomputed features?

When `computed_properties` is set in `GuacaMolConfig`, each molecule's physicochemical descriptors (e.g., `MolWt`, `MolLogP`, `QED`) are computed once via RDKit and stored in `candidate.features`. The `MLPModel` reads these precomputed values directly from `candidate.features` rather than recomputing fingerprints on-the-fly, making training and inference significantly faster.

### Experiment Overview

In this tutorial, we will:
1. Load the GuacaMol dataset with a cold-start `low_vs_high` split
2. Inspect the initial training set to confirm the cold-start
3. Instantiate an MLP surrogate that uses precomputed physicochemical features
4. Run 5 rounds of greedy active learning
5. Visualise how QED scores of acquired candidates improve across rounds

### Framework Components

Before we start, let's understand the key components of the ALF framework used in this notebook:

1. **Dataset** ([`GuacaMol`](https://instadeepai.github.io/alf/api/alf_tools/datasets/guacamol/)): Loads SMILES strings, computes RDKit properties, and handles data splitting into train/validation/test/candidate_pool sets using the `low_vs_high` strategy.

2. **Surrogate Model** ([`MLPModel`](https://instadeepai.github.io/alf/api/alf_tools/models/mlp/)): A feed-forward MLP trained on precomputed physicochemical features stored in `candidate.features`. Architecture: Linear → GELU → LayerNorm → Dropout, repeated per hidden layer, followed by a scalar output head.

3. **Search Strategy** ([`DatasetSearch`](https://instadeepai.github.io/alf/api/alf_core/optimizer/search/)): Searches within the candidate pool (the unlabelled portion of the dataset). The pool shrinks by `batch_size` each round as molecules are acquired.

4. **Acquisition Function** ([`Greedy`](https://instadeepai.github.io/alf/api/alf_tools/optimizer/acquisition_functions/greedy/)): Selects molecules with the highest predicted QED score. Simple but effective for demonstrating cold-start recovery.

5. **Optimizer** ([`Optimizer`](https://instadeepai.github.io/alf/api/alf_core/optimizer/optimizer/)): Combines the acquisition function and search strategy, handling the ask (select candidates) / tell (update with oracle results) cycle.

6. **Oracle** ([`Oracle`](https://instadeepai.github.io/alf/api/alf_core/oracle/)): Provides ground-truth QED labels from the dataset, simulating an expensive experimental evaluation.

7. **Task** ([`DesignTask`](https://instadeepai.github.io/alf/api/alf_core/tasks/design_task/)): Orchestrates the complete active learning loop across multiple acquisition rounds.

### Step 0: Environment Setup

First, ensure the ALF packages and their dependencies (including RDKit) are installed:


In [ ]:
# import sys
# !uv pip install -e . --python {sys.executable}

### Step 1: Import Required Libraries

Let's import all the necessary components from the ALF framework:


In [ ]:
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from alf_core import (
    DatasetSearch,
    DesignTask,
    FileStateLogger,
    Optimizer,
    Oracle,
    Surrogate,
    TerminalStateLogger,
)
from alf_tools.datasets import GuacaMol, GuacaMolConfig
from alf_tools.models import MLPModel, MLPModelConfig, MLPTrainConfig
from alf_tools.optimizer.acquisition_functions import Greedy

print("All imports successful!")

### Step 2: Load the Dataset with `low_vs_high` Split

We configure the GuacaMol dataset with:
- `target_property="QED"`: predict Quantitative Estimate of Drug-likeness (range 0–1, higher is more drug-like)
- `split_mode="low_vs_high"`: seeds the training set with only low-QED molecules (cold start)
- `computed_properties=["MolWt", "MolLogP", "QED"]`: precompute these RDKit properties and store them in `candidate.features` for fast MLP featurisation
- `max_molecules=1000`: cap the corpus at 1 000 molecules for tutorial speed


In [ ]:
config = GuacaMolConfig(
    name="guacamol_offline_tutorial",
    modality="sequence",
    seed=42,
    target_property="QED",
    split_mode="low_vs_high",
    computed_properties=["MolWt", "MolLogP", "QED"],
    max_molecules=10_000,
    train_ratio=0.1,
    validation_frac=0.1,
    test_ratio=0.2,
)
dataset = GuacaMol(config)

print("Dataset initialised!")
print(f"Total molecules:      {len(dataset._raw_dataset)}")
print(f"Training set size:    {len(dataset.train_dataset)}")
print(f"Validation set size:  {len(dataset.validation_dataset)}")
print(f"Test set size:        {len(dataset.test_dataset)}")
print(f"Candidate pool size:  {len(dataset.candidate_pool)}")

# Inspect initial training labels — should be entirely low-QED values
train_labels = dataset.train_dataset.labels
all_labels = dataset._raw_dataset.labels
all_median = float(np.median(all_labels))

print("\nCold-start verification (low_vs_high split):")
print(f"  Corpus QED median:          {all_median:.4f}")
print(f"  Initial train QED max:      {train_labels.max():.4f}")
print(f"  Initial train QED mean:     {train_labels.mean():.4f}")
print()
print("  => All initial training molecules have QED < corpus median.")
print("  => The model must learn to find high-QED molecules from a low-QED start.")

### Step 3: Instantiate the MLP Surrogate

The `MLPModel` uses a feed-forward network with the following featurisation strategy:

- **Precomputed path** (used here): when `candidate.features` contains numeric values (i.e., the properties we listed in `computed_properties`), the model reads those values directly. Here each molecule is represented by a 3-dimensional vector: `[MolWt, MolLogP, QED]`, sorted by key name.
- **SMILES fallback**: when `candidate.features` has no numeric values, the model computes an ECFP4 fingerprint (2 048 bits) plus 9 physicochemical descriptors on-the-fly via RDKit.

Using precomputed features is significantly faster because the RDKit computation happens once at dataset load time rather than at every training step.

Architecture: `Linear(3→128) → GELU → LayerNorm → Dropout → Linear(128→64) → GELU → LayerNorm → Dropout → Linear(64→1)`


In [ ]:
mlp_model = MLPModel(
    model_config=MLPModelConfig(hidden_dims=[128, 64]),
    train_config=MLPTrainConfig(num_epochs=50, log_frequency=10),
)
surrogate = Surrogate(model=mlp_model)

print("MLP surrogate initialised!")
print("Model config:")
print(f"  Hidden dims:   {mlp_model.model_config.hidden_dims}")
print(f"  Dropout:       {mlp_model.model_config.dropout}")
print("Training config:")
print(f"  Epochs:        {mlp_model.train_config.num_epochs}")
print(f"  Batch size:    {mlp_model.train_config.batch_size}")
print(f"  Learning rate: {mlp_model.train_config.learning_rate}")
print(f"  Log frequency: {mlp_model.train_config.log_frequency}")
print()
print("Featurisation: reads precomputed [MolLogP, MolWt, QED] from candidate.features")

### Step 4: Set Up the Acquisition Strategy, Optimizer, and Oracle

We use a **greedy** acquisition function (select molecules with the highest predicted QED) and a **dataset search** strategy (search within the candidate pool).


In [ ]:
# Acquisition function: greedy selection of highest predicted QED
acquisition_fn = Greedy()

# Search strategy: enumerate the unlabelled candidate pool
search_fn = DatasetSearch()

# Combine into optimizer
optimizer = Optimizer(acquisition_fn=acquisition_fn, search_fn=search_fn)

# Oracle: return ground-truth QED labels from the dataset
oracle = Oracle(scorer=dataset)

print("Components initialised!")
print("  Acquisition function: Greedy (select highest predicted QED)")
print("  Search strategy:      DatasetSearch (candidate pool)")
print("  Oracle:               GuacaMol dataset QED labels")

### Step 5: Configure and Run the Design Task

We run 5 acquisition rounds, acquiring 50 molecules per round. The pool starts with ~700 molecules and shrinks by 50 each round.


In [ ]:
import logging

logging.basicConfig(level=logging.INFO)

# Set up loggers
terminal_logger = TerminalStateLogger()
save_path = Path("results/guacamol_offline_design/")
if save_path.exists():
    shutil.rmtree(save_path)
file_logger = FileStateLogger(output_path=save_path)
loggers = [terminal_logger, file_logger]

# Configure the design task
task = DesignTask(num_acq_rounds=5, acq_batch_size=50)

# Set up the initial state
print("Setting up experiment...")
state = task.setup(dataset=dataset, surrogate=surrogate)

print("Initial state:")
print(f"  Training samples:   {len(state.dataset.train_dataset)}")
print(f"  Validation samples: {len(state.dataset.validation_dataset)}")
print(f"  Test samples:       {len(state.dataset.test_dataset)}")
print(f"  Candidate pool:     {len(state.dataset.candidate_pool)}")

# Run the active learning experiment
print("\nStarting GuacaMol offline active learning experiment...")
print("=" * 60)

task.run(state, state_loggers=loggers, optimizer=optimizer, oracle=oracle)

print("\nExperiment completed!")

### Step 6: Analyse the Results

We load the saved metrics and visualise how QED scores of acquired candidates improve across rounds.

Because `split_mode="low_vs_high"` starts the training set with only low-QED molecules, early rounds should return low-to-moderate QED candidates. As the surrogate model learns the QED landscape from these labels (and the oracle augments the training set with newly acquired molecules), later rounds should discover progressively higher-QED candidates.

We track:
- **Round Mean QED**: average QED of the batch acquired in each round
- **Round Max QED**: best QED molecule found per round
- **Top 10% Recall**: fraction of acquired molecules that belong to the top 10% of all candidates by QED
- **Surrogate Spearman**: rank correlation between predicted and true QED on the test set


In [ ]:
# Load metrics from CSV
metrics = pd.read_csv(save_path / "metrics.csv")

print("Experiment Summary:")
print(f"Total rounds (including zeroth round): {len(metrics)}")
print(f"Initial train set mean QED:  {metrics['dataset/train_mean'].iloc[0]:.4f}")
print(f"Final acquired batch mean QED: {metrics['acquired_candidates/round_mean'].iloc[-1]:.4f}")
print(f"Best QED found:              {metrics['acquired_candidates/round_max'].max():.4f}")
print(f"Final surrogate Spearman:    {metrics['surrogate/test_spearman'].iloc[-1]:.4f}")

In [ ]:
# Plot results — exclude the zeroth (initial training) round for per-round acquisition metrics
metrics = metrics.reset_index().rename(columns={"index": "round"})
metrics_plot = metrics.iloc[1:]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle(
    "GuacaMol QED Offline Design — Cold-Start Active Learning Results",
    fontsize=15,
    fontweight="bold",
)

rounds = metrics_plot["round"]

# 1. Round Mean QED
axes[0, 0].plot(
    rounds,
    metrics_plot["acquired_candidates/round_mean"],
    marker="o",
    linewidth=2,
    markersize=8,
    color="#e74c3c",
)
# Add initial training mean as a reference line
axes[0, 0].axhline(
    y=metrics["dataset/train_mean"].iloc[0],
    linestyle="--",
    color="gray",
    linewidth=1,
    label=f"Initial train mean ({metrics['dataset/train_mean'].iloc[0]:.3f})",
)
axes[0, 0].set_xlabel("Round", fontsize=12)
axes[0, 0].set_ylabel("Mean QED", fontsize=12)
axes[0, 0].set_title("Acquired Batch Mean QED", fontsize=13, fontweight="bold")
axes[0, 0].legend(fontsize=9)
axes[0, 0].grid(True, alpha=0.3)

# 2. Top 10% Recall
axes[0, 1].plot(
    rounds,
    metrics_plot["optimizer/top_10pc_recall"],
    marker="^",
    linewidth=2,
    markersize=8,
    color="#2ecc71",
)
axes[0, 1].set_xlabel("Round", fontsize=12)
axes[0, 1].set_ylabel("Top 10% Recall", fontsize=12)
axes[0, 1].set_title("Top 10% Recall in Acquired Batch", fontsize=13, fontweight="bold")
axes[0, 1].set_ylim([0, 1])
axes[0, 1].grid(True, alpha=0.3)

# 3. Top 100 n Recall
axes[0, 2].plot(
    rounds,
    metrics_plot["optimizer/top_100_recall"],
    marker="^",
    linewidth=2,
    markersize=8,
    color="#2ecc71",
)
axes[0, 2].set_xlabel("Round", fontsize=12)
axes[0, 2].set_ylabel("Top 100 n Recall", fontsize=12)
axes[0, 2].set_title("Top 100 n Recall in Acquired Batch", fontsize=13, fontweight="bold")
axes[0, 2].set_ylim([0, 1])
axes[0, 2].grid(True, alpha=0.3)

# 4. Round Max QED
axes[1, 0].plot(
    rounds,
    metrics_plot["acquired_candidates/round_max"],
    marker="D",
    linewidth=2,
    markersize=8,
    color="#9b59b6",
)
axes[1, 0].set_xlabel("Round", fontsize=12)
axes[1, 0].set_ylabel("Max QED", fontsize=12)
axes[1, 0].set_title("Acquired Batch Max QED", fontsize=13, fontweight="bold")
axes[1, 0].grid(True, alpha=0.3)

# 5. Surrogate Spearman Correlation
axes[1, 1].plot(
    metrics["round"],
    metrics["surrogate/test_spearman"],
    marker="v",
    linewidth=2,
    markersize=8,
    color="#34495e",
)
axes[1, 1].set_xlabel("Round", fontsize=12)
axes[1, 1].set_ylabel("Spearman Correlation", fontsize=12)
axes[1, 1].set_title("Surrogate Prediction Quality (Test Spearman)", fontsize=13, fontweight="bold")
axes[1, 1].grid(True, alpha=0.3)

axes[1, 2].axis("off")  # Hide unused subplot

plt.tight_layout()
plt.show()

In [ ]:
# Optionally, clean up the results directory
if save_path.exists():
    shutil.rmtree(save_path)

print("Results directory cleaned up!")

## Conclusion

This tutorial demonstrated an **offline design experiment** on the GuacaMol QED property using the ALF framework, with several features worth highlighting:

### Cold-start via `split_mode="low_vs_high"`

The `low_vs_high` split placed only low-QED molecules in the initial training set and high-QED molecules in the candidate pool. This is a realistic and challenging scenario — the surrogate model has to generalise well beyond its training distribution to find high-QED candidates. As active learning progresses and oracle labels are collected, the training set becomes richer, and the surrogate should improve its ability to identify high-QED regions of chemical space.

### Precomputed features via `computed_properties`

By specifying `computed_properties=["MolWt", "MolLogP", "QED"]`, RDKit descriptors are computed once at dataset load time and stored in `candidate.features`. The `MLPModel` reads these values directly, avoiding redundant computation during training and inference. This is particularly beneficial when the candidate pool is large.

### Shrinking candidate pool

Each acquisition round removes `batch_size=50` molecules from the candidate pool. After 5 rounds, 250 molecules have been acquired and labelled. In a real experiment with a larger pool, this gradual exhaustion is not a concern — but for tutorial purposes it illustrates the offline design loop clearly.

### Next steps

- Try `split_mode="random"` to compare against a standard active learning baseline without a cold start.
- Increase `max_molecules` (e.g., to 10 000 or the full corpus) for more realistic benchmarking.
- Swap `Greedy` for an uncertainty-aware acquisition function (e.g., UCB) to explore vs. exploit trade-offs.
- Target a different property (e.g., `MolWt`, `MolLogP`) by changing `target_property` in `GuacaMolConfig`.

**Happy designing!**